# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damiannogueira/flyrank-ml-intership/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice

I will start with **Logistic Regression** as the first learned model.

The prediction target is binary: whether a page shows an observed future decline according to the outcome definition established in the previous assignments. However, the model will be used as a **ranking model** rather than only as a yes/no classifier. Its predicted probability will become the score used to rank pages for review.

Logistic Regression is a useful first model because it is simple, reproducible, and relatively easy to interpret. This makes it a good learned comparison against the transparent Week-4 rule before adding more complex models.

The model will use only pre-decision signals:

- `impressions_21d`: search visibility during the first 21 days.
- `clicks_21d`: observed Google Search clicks during the same period.
- `ctr_21d`: clicks divided by impressions during the pre-decision window.
- `avg_position_21d`: weighted average Google Search position during the pre-decision window.
- `active_days_21d`: number of observed active days during the pre-decision period.

`client_hash_id` and `content_hash_id` will be retained only for grouping, splitting, and inspecting errors. They will not be model features.

The future outcome and any variables derived from the future window will be used only as labels for evaluation and will never be supplied to the model as inputs.

The main evaluation metric will remain **Precision@50**, matching the ranking objective established earlier. The Week-4 baseline and the learned model must be evaluated on the same evaluation rows and with the same metric for the comparison to be meaningful.

In [1]:
# Define the modeling choices before training anything.
# These constants make the experiment easier to read and reproduce.

MODEL_NAME = "Logistic Regression"
TARGET = "future_decline_target"

FEATURES = [
    "impressions_21d",
    "clicks_21d",
    "ctr_21d",
    "avg_position_21d",
    "active_days_21d",
]

# Precision@50 matches the ranking objective defined in the earlier assignments.
K = 50

# Fixed seed for reproducible train/test operations when randomness is involved.
RANDOM_STATE = 42

print("Model:", MODEL_NAME)
print("Target:", TARGET)
print("Features:", FEATURES)
print("Ranking metric: Precision@", K, sep="")
print("Random state:", RANDOM_STATE)

Model: Logistic Regression
Target: future_decline_target
Features: ['impressions_21d', 'clicks_21d', 'ctr_21d', 'avg_position_21d', 'active_days_21d']
Ranking metric: Precision@50
Random state: 42


## 2. Split design

I will use a **client-grouped train/test split**.

Pages from the same client may share content, traffic, and measurement patterns. If pages from one client appeared in both training and evaluation data, the model could benefit from client-specific patterns and produce an overly optimistic result.

To reduce this risk, all pages belonging to a given `client_hash_id` will stay entirely on one side of the split.

The pre-decision features use only March 1–21, 2026. The observed outcome is measured later, during March 22–31, 2026. This preserves the direction of time: future information is used only to define the target and never as a model input.

I will reserve approximately 20% of clients for evaluation and use a fixed `random_state=42` so the split is reproducible.

The Week-4 baseline and the learned model will both be evaluated on these same held-out rows using the same Precision@50 metric.

In [2]:
# Prepare the March 2026 modeling dataset and create an honest
# client-grouped train/test split.

%pip -q install duckdb scikit-learn

from google.colab import userdata
import duckdb
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit


# Load the Hugging Face token securely from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)


# March 2026 is our mid-panel development month.
MARCH_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""


# Build one row per page.
#
# Features: March 1-21 only.
# Outcome:  March 22-31 only.
model_frame = con.sql(f"""
    WITH pre_decision AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS impressions_21d,
            SUM(gsc_clicks) AS clicks_21d,

            SUM(gsc_sum_position)
                / NULLIF(SUM(gsc_impressions), 0)
                AS avg_position_21d,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_impressions > 0 THEN report_date
                END
            ) AS active_days_21d

        FROM {MARCH_REL}

        WHERE
            gsc_data_available IS TRUE
            AND report_date
                BETWEEN DATE '2026-03-01'
                AND DATE '2026-03-21'

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING SUM(gsc_impressions) > 0
    ),

    later_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future_impressions_10d

        FROM {MARCH_REL}

        WHERE
            gsc_data_available IS TRUE
            AND report_date
                BETWEEN DATE '2026-03-22'
                AND DATE '2026-03-31'

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.impressions_21d,
        p.clicks_21d,
        p.avg_position_21d,
        p.active_days_21d,
        f.future_impressions_10d

    FROM pre_decision AS p

    INNER JOIN later_window AS f
        USING (client_hash_id, content_hash_id)
""").df()


# CTR is stored as a percentage:
# clicks divided by impressions, multiplied by 100.
model_frame["ctr_21d"] = (
    100.0
    * model_frame["clicks_21d"]
    / model_frame["impressions_21d"]
)


# Compare daily impression rates because the two windows
# contain different numbers of days: 21 versus 10.
model_frame["future_change_pct"] = (
    (
        model_frame["future_impressions_10d"] / 10
        - model_frame["impressions_21d"] / 21
    )
    / (model_frame["impressions_21d"] / 21)
    * 100
)


# Target = 1 when the later daily impression rate
# is more than 20% below the pre-decision rate.
model_frame[TARGET] = (
    model_frame["future_change_pct"] < -20
).astype(int)


# Sort first so the experiment starts from a deterministic order.
model_frame = model_frame.sort_values(
    ["client_hash_id", "content_hash_id"]
).reset_index(drop=True)


# Keep entire clients on only one side of the split.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(
        model_frame,
        model_frame[TARGET],
        groups=model_frame["client_hash_id"],
    )
)

train_frame = model_frame.iloc[train_idx].copy()
test_frame = model_frame.iloc[test_idx].copy()


# Verify that no client appears in both datasets.
train_clients = set(train_frame["client_hash_id"])
test_clients = set(test_frame["client_hash_id"])

client_overlap = train_clients.intersection(test_clients)


print(f"Total pages: {len(model_frame):,}")
print(f"Train pages: {len(train_frame):,}")
print(f"Test pages: {len(test_frame):,}")

print()

print(f"Train clients: {len(train_clients)}")
print(f"Test clients: {len(test_clients)}")
print(f"Client overlap: {len(client_overlap)}")

print()

print(
    "Train future-decline rate:",
    f"{train_frame[TARGET].mean():.1%}"
)

print(
    "Test future-decline rate:",
    f"{test_frame[TARGET].mean():.1%}"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages: 145,091
Train pages: 136,239
Test pages: 8,852

Train clients: 35
Test clients: 9
Client overlap: 0

Train future-decline rate: 29.9%
Test future-decline rate: 29.9%


## 3. Train + compare vs my baseline

### Training and comparison

I will train the Logistic Regression model using only the pre-decision features defined earlier.

Because the feature scales are very different — for example, impressions can reach large values while CTR and average position are much smaller — I will standardize the numeric features inside a scikit-learn pipeline before fitting Logistic Regression.

The Week-4 baseline will be recreated on the same held-out test rows using the same rule:

- at least 500 pre-decision impressions;
- average search position from 4 up to, but not including, 20;
- qualifying pages ranked by pre-decision impressions.

The learned model and the baseline will therefore be compared on the **same evaluation data** using **Precision@50**. I will also report the test base rate so the ranking results have context.

In [3]:
# Train Logistic Regression and compare it with the Week-4 baseline
# on exactly the same held-out test rows.

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# Separate model inputs from the target.
X_train = train_frame[FEATURES]
y_train = train_frame[TARGET]

X_test = test_frame[FEATURES]
y_test = test_frame[TARGET]


# StandardScaler learns scaling parameters from train only.
# LogisticRegression then learns the relationship between
# the pre-decision features and the observed future target.
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

model.fit(X_train, y_train)


# Probability of the positive class:
# future_decline_target = 1.
model_scores = model.predict_proba(X_test)[:, 1]


# Create a deterministic evaluation frame.
evaluation = test_frame[
    [
        "client_hash_id",
        "content_hash_id",
        TARGET,
        "impressions_21d",
        "avg_position_21d",
    ]
].copy()

evaluation["model_score"] = model_scores


# Recreate the Week-4 baseline rule on the same test rows.
evaluation["baseline_qualifies"] = (
    (evaluation["impressions_21d"] >= 500)
    & (evaluation["avg_position_21d"] >= 4)
    & (evaluation["avg_position_21d"] < 20)
)

evaluation["baseline_score"] = (
    evaluation["baseline_qualifies"].astype(int)
    * evaluation["impressions_21d"]
)


def precision_at_k(frame, score_column, k=50, eligible_column=None):
    """
    Rank rows by score and return Precision@K.

    If eligible_column is supplied, only rows selected by the
    baseline rule are allowed into that ranking.
    """

    ranked = frame.copy()

    if eligible_column is not None:
        ranked = ranked[ranked[eligible_column]].copy()

    ranked = ranked.sort_values(
        [score_column, "client_hash_id", "content_hash_id"],
        ascending=[False, True, True],
        kind="stable",
    )

    top_k = ranked.head(k)

    return top_k[TARGET].mean(), len(top_k)


# Evaluate the learned model.
model_p50, model_n = precision_at_k(
    evaluation,
    score_column="model_score",
    k=K,
)


# Evaluate the frozen Week-4 baseline.
baseline_p50, baseline_n = precision_at_k(
    evaluation,
    score_column="baseline_score",
    k=K,
    eligible_column="baseline_qualifies",
)


# Base rate = prevalence of the positive target in the held-out test data.
base_rate = y_test.mean()


comparison = pd.DataFrame(
    {
        "method": [
            "Test base rate",
            "Week-4 baseline",
            "Logistic Regression",
        ],
        "precision_at_50": [
            base_rate,
            baseline_p50,
            model_p50,
        ],
        "rows_in_top_k": [
            len(y_test),
            baseline_n,
            model_n,
        ],
    }
)

comparison["precision_at_50"] = (
    comparison["precision_at_50"] * 100
).round(1)

comparison


,method,precision_at_50,rows_in_top_k
0,Test base rate,29.9,8852
1,Week-4 baseline,20.0,50
2,Logistic Regression,28.0,50


### Comparison result

On the held-out client-grouped test set, Logistic Regression reached **28.0% Precision@50**, compared with **20.0%** for the Week-4 rule baseline.

The learned model therefore improved on the hand-written baseline by 8 percentage points at the top 50.

However, the test-set positive base rate is **29.9%**, which is slightly higher than the model's Precision@50. This means the model did not concentrate future-decline cases above their overall prevalence in the held-out data.

The result is therefore mixed: Logistic Regression performs better than the Week-4 rule on the same rows and metric, but the current ranking is not yet strong enough to claim useful lift over the test base rate.

## 4. Errors and interpretation

### Error analysis

I will inspect the standardized Logistic Regression coefficients to understand which pre-decision signals the model relies on most.

The coefficient signs and magnitudes describe associations learned by this model; they are not causal effects.

I will also inspect false positives among the model's highest-ranked pages. These are pages the model ranked highly for future decline but whose observed target was actually 0. Looking at these cases helps identify where the ranking can be misleading.

In [4]:
# Inspect what the model relies on and review concrete ranking errors.

# Extract the fitted Logistic Regression model from the pipeline.
logistic_model = model.named_steps["logistic_regression"]

# Because the inputs were standardized, coefficient magnitudes
# are more directly comparable across the five features.
coefficient_table = pd.DataFrame(
    {
        "feature": FEATURES,
        "coefficient": logistic_model.coef_[0],
    }
)

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "absolute_coefficient",
    ascending=False,
).reset_index(drop=True)

print("Model coefficients:")
display(coefficient_table)


# Rank the held-out pages exactly as the model ranks them.
model_ranking = evaluation.sort_values(
    ["model_score", "client_hash_id", "content_hash_id"],
    ascending=[False, True, True],
    kind="stable",
).reset_index(drop=True)

model_ranking.insert(
    0,
    "model_rank",
    range(1, len(model_ranking) + 1),
)


# False positives in the top 50:
# highly ranked pages whose observed future target was actually 0.
top50 = model_ranking.head(K).copy()

false_positives_top50 = top50[
    top50[TARGET] == 0
].copy()

print()
print(f"False positives in top {K}: {len(false_positives_top50)}")
print(
    f"True positives in top {K}: "
    f"{(top50[TARGET] == 1).sum()}"
)


# Show three concrete wrong cases with their pre-decision signals.
wrong_cases = false_positives_top50.head(3)[
    [
        "model_rank",
        "content_hash_id",
        "model_score",
        "impressions_21d",
        "avg_position_21d",
        TARGET,
    ]
].copy()

# Bring the remaining model features back for interpretation.
wrong_cases = wrong_cases.merge(
    test_frame[
        [
            "content_hash_id",
            "clicks_21d",
            "ctr_21d",
            "active_days_21d",
        ]
    ],
    on="content_hash_id",
    how="left",
)

wrong_cases


Model coefficients:


,feature,coefficient,absolute_coefficient
0,clicks_21d,-0.877076,0.877076
1,active_days_21d,0.337303,0.337303
2,impressions_21d,0.142123,0.142123
3,avg_position_21d,-0.040549,0.040549
4,ctr_21d,0.007574,0.007574



False positives in top 50: 36
True positives in top 50: 14


,model_rank,content_hash_id,model_score,impressions_21d,avg_position_21d,future_decline_target,clicks_21d,ctr_21d,active_days_21d
0,1,content_1bc14f25ea5989e0,0.434111,11820.0,8.110152,0,4.0,0.033841,21
1,2,content_31ea2c45cdc6c46e,0.408457,5482.0,10.909887,0,1.0,0.018242,21
2,4,content_fbe9d7902c683443,0.398929,4175.0,7.157126,0,1.0,0.023952,21


### Error interpretation

The model relies most strongly on `clicks_21d`. Its standardized coefficient is negative (`-0.877`), meaning that, within this fitted model, pages with more pre-decision clicks tend to receive a lower future-decline score when the other included features are held constant.

`active_days_21d` (`+0.337`) and `impressions_21d` (`+0.142`) are the next strongest coefficients. `avg_position_21d` has a much smaller contribution, while `ctr_21d` is close to neutral in this fit.

These coefficients describe associations learned from the training data, not causal effects. They should also be interpreted cautiously because clicks, impressions, and CTR are related measurements.

The top-50 ranking contains **14 true positives and 36 false positives**, consistent with the observed 28.0% Precision@50.

The three inspected false positives show a similar pattern: all were active on all 21 pre-decision days, had thousands of search impressions, but received very few clicks and extremely low CTRs. This combination matches the signals that push the model toward a higher decline score, especially the strong negative coefficient on clicks.

However, none of these three pages showed the defined future decline. This indicates that sustained visibility combined with very low click volume is not sufficient by itself to identify future decline reliably.

The highest-ranked model scores are also below 0.5. This is not a contradiction because the model is being used for **relative ranking**, not threshold-based classification. Precision@50 depends on the order of the scores, not on whether each probability exceeds 0.5.

Overall, the error analysis supports the earlier comparison result: the learned model improves on the Week-4 rule, but its top-ranked recommendations still contain many false positives and do not yet provide lift over the held-out base rate.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.